# Week 1 Advanced — Motor Speed Control as an Engineering Problem

The primer showed that PI removes steady-state error. Here we add the things that make real control harder: torque saturation, load disturbances, anti-windup, and bandwidth tradeoffs.

Plant: $$J\dot{\omega}=T_m-T_L-b\omega$$
Controller: $$T_m=K_p e+K_i\int e\,dt$$
with actuator limit $$|T_m|\le T_{max}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

J, b, Tmax = 0.08, 0.03, 3.0
dt = 1e-3
t = np.arange(0, 8, dt)


## 1. Baseline PI versus saturated PI

A controller can demand physically impossible torque. Once the actuator saturates, the integrator may continue accumulating error and create **integrator windup**.

In [ ]:
def simulate(kp, ki, anti_windup=False):
    omega = 0.0
    integ = 0.0
    W, U, E = [], [], []
    for ti in t:
        ref = 40.0 if ti < 5.0 else 15.0
        load = 0.3 if ti < 2.5 else 1.2
        e = ref - omega
        u_unsat = kp*e + ki*integ
        u = np.clip(u_unsat, -Tmax, Tmax)
        if anti_windup:
            if abs(u_unsat) < Tmax or np.sign(e) != np.sign(u_unsat):
                integ += e*dt
        else:
            integ += e*dt
        omega += (u-load-b*omega)/J*dt
        W.append(omega); U.append(u); E.append(e)
    return np.array(W), np.array(U), np.array(E)

w_bad, u_bad, _ = simulate(0.35, 1.5, False)
w_aw, u_aw, _ = simulate(0.35, 1.5, True)
plt.figure(figsize=(9,4))
plt.plot(t, w_bad, label='PI, no anti-windup')
plt.plot(t, w_aw, label='PI + anti-windup')
plt.axvline(2.5, ls=':', label='load step')
plt.axvline(5.0, ls='--', label='command drop')
plt.xlabel('Time [s]'); plt.ylabel('Speed [rad/s]'); plt.grid(True); plt.legend(); plt.show()


## 2. Engineering questions

1. Why does saturation barely matter during small disturbances but dominate large command changes?
2. Increase $K_i$ by 4x. What happens after the command drops at 5 s?
3. Reduce $T_{max}$ to 1.5. Can your original settling-time requirement still be achieved?
4. Estimate the steady-state torque required at 40 rad/s with the heavy load before simulating it.

**Deliverable:** choose gains and a torque limit, report rise time, settling time, overshoot, maximum torque, and disturbance-recovery time.